### PyTorch Computational Graph Tutorial

**1. Introduction**

What is a computational graph?
- A computational graph is a structure that represents the sequence of operations performed on tensors.
- PyTorch uses a dynamic computational graph, which is created on-the-fly during the forward pass.
- The computational graph is essential for autograd, PyTorch's automatic differentiation engine.
- Autograd is used to compute the gradients of the loss function with respect to the model parameters.
- The gradients are then used to update the model parameters using an optimization algorithm (e.g. stochastic gradient descent).

In this tutorial we will play around with PyTorch's computational graph and autograd functionality.

#### Some useful videos:

- [PyTorch Autograd Explained - In-depth Tutorial (**only ~14 minutes**)](https://www.youtube.com/watch?v=MswxJw-8PvE)
- [Computation Graph (C1W2L07)](https://www.youtube.com/watch?v=hCP1vGoCdYU&list=PLkDaE6sCZn6Ec-XTbcX1uRg2_u4xOEky0&index=13)
- [Derivatives With Computation Graphs (C1W2L08)](https://www.youtube.com/watch?v=nJyUyKN-XBQ&list=PLkDaE6sCZn6Ec-XTbcX1uRg2_u4xOEky0&index=14)
- [Deep L-Layer Neural Network (C1W4L01)](https://www.youtube.com/watch?v=2gw5tE2ziqA&list=PLkDaE6sCZn6Ec-XTbcX1uRg2_u4xOEky0&index=36)
- [Forward Propagation in a Deep Network (C1W4L02)](https://www.youtube.com/watch?v=a8i2eJin0lY&list=PLkDaE6sCZn6Ec-XTbcX1uRg2_u4xOEky0&index=39)

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

### 2. Setting Up Tensors and Building a Simple Computational Graph

We create two tensors `x` and `y` and perform a simple operation on them to create a third tensor `z`. The operation is $z=xy+y^2$.

When then compute the gradients of `z` with respect to `x` and `y` using autograd. We do this by calling the `backward()` method on `z`.

The partial derivatives of `z` with respect to `x` and `y` are stored in the `grad` attribute of `x` and `y`.

We have that 
$$\frac{\partial z}{\partial x} = y$$
and
$$\frac{\partial z}{\partial y} = x + 2y$$

So with $x=2$ and $y=3$ we should get $\frac{\partial z}{\partial x} = 3$ and $\frac{\partial z}{\partial y} = 8$.

In [2]:
# Create tensors
x = torch.tensor(2.0, requires_grad=True)  # Enable gradient tracking
y = torch.tensor(3.0, requires_grad=True)

# Define a simple function
z = x * y + y**2

# Print the result
print(f"z = {z}")

# Perform backpropagation
z.backward()

# Access the gradients
print(f"Gradient of x: {x.grad}")
print(f"Gradient of y: {y.grad}")

z = 15.0
Gradient of x: 3.0
Gradient of y: 8.0


## Breaking the computational graph

Now, we look at pitfalls when using autograd.

### In-place operations on tensors reqruiring gradients

In [3]:
x = torch.tensor(2.0, requires_grad=True)

try:
    x += 1  # This will throw an error
except RuntimeError as e:
    print(f"In-place operation error: {e}")

In-place operation error: a leaf Variable that requires grad is being used in an in-place operation.


In [4]:
x = torch.tensor(2.0, requires_grad=True)
x = x + 1  # This is fine as it creates a new tensor

### Detaching tensors from the computational graph

If we detach the tensor $y$ and then let $z=y^2$, and then try to compute the gradients of `z` with respect to `y`, we will get an error.

In [5]:
y = torch.tensor(3.0, requires_grad=True)

y_detached = y.detach() # Detach the tensor from the computation graph
z_detached = y_detached ** 2
print(f"z_detached.requires_grad? {z_detached.requires_grad}")
try:
    z_detached.backward()
except RuntimeError as e:
    print(f"Detach error: {e}")

z_detached.requires_grad? False
Detach error: element 0 of tensors does not require grad and does not have a grad_fn


### Using `with torch.no_grad()` incorrectly

Sometimes you want to use `with torch.no_grad()` to avoid tracking gradients as this can save memory and speed up computations. Typically you would use `with torch.no_grad()` when you are evaluating the model on the validation or test set and do not need to compute gradients. Or if you are manually updating the model parameters using the gradients and do not want to track gradients during this process.

However, if you use `with torch.no_grad()` incorrectly, you might end up with a computational graph that is not connected to the tensors you want to compute gradients with respect to!

In [ ]:
x = torch.tensor(2.0, requires_grad=True)  # Enable gradient tracking
y = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y_no_grad = x * y

print(f"y_no_grad.requires_grad? {y_no_grad.requires_grad}")

try:
    y_no_grad.backward()
except RuntimeError as e:
    print(f"No grad error: {e}")

y_no_grad.requires_grad? False
No grad error: element 0 of tensors does not require grad and does not have a grad_fn


## A small neural network

We build a small neural network with two layers with ReLU activation function between them:

`Input -> Linear -> ReLU -> Linear -> Output` 

where the input is a tensor of size 2 and the output is a tensor of size 1. In total we have 13 learnable parameters in our model.

In [7]:
# Input tensor
data = torch.tensor([[-1.0, 2.0]], requires_grad=True)

# Define the network
model = nn.Sequential(
    nn.Linear(2, 3),  # Linear layer with 2 inputs and 3 outputs
    nn.ReLU(),        # ReLU activation
    nn.Linear(3, 1)   # Linear layer with 3 inputs and 1 output
)

print("Our model with randomly initialized weights:")
for name, param in model.named_parameters():
    print(f"Layer: {name}\n\tSize: {param.size()}\n\tValues : {param} \n")

# Forward pass
output = model(data)
print(f"Output: {output}")

# Compute gradients
output.backward()

# Access gradients for parameters
for name, param in model.named_parameters():
    print(f"Gradient for {name}: {param.grad}")

Our model with randomly initialized weights:
Layer: 0.weight
	Size: torch.Size([3, 2])
	Values : Parameter containing:
tensor([[-0.0053,  0.3793],
        [-0.5820, -0.5204],
        [-0.2723,  0.1896]], requires_grad=True) 

Layer: 0.bias
	Size: torch.Size([3])
	Values : Parameter containing:
tensor([-0.0140,  0.5607, -0.0628], requires_grad=True) 

Layer: 2.weight
	Size: torch.Size([1, 3])
	Values : Parameter containing:
tensor([[ 0.1528, -0.1745, -0.1135]], requires_grad=True) 

Layer: 2.bias
	Size: torch.Size([1])
	Values : Parameter containing:
tensor([-0.5516], requires_grad=True) 

Output: tensor([[-0.5216]], grad_fn=<AddmmBackward0>)
Gradient for 0.weight: tensor([[-0.1528,  0.3055],
        [ 0.1745, -0.3490],
        [ 0.1135, -0.2270]])
Gradient for 0.bias: tensor([ 0.1528, -0.1745, -0.1135])
Gradient for 2.weight: tensor([[0.7499, 0.1019, 0.5888]])
Gradient for 2.bias: tensor([1.])


In the example above, explain why the gradient of the output with respect to the bias in the output layer is `1.0`.

Trying to modify the model parameters in-place will result in an error.

In [8]:
# Perform an in-place operation on the model parameters (requiring grad)
try:
    model[0].weight += 1  # In-place operation on model parameters
except RuntimeError as e:
    print(f"In-place operation error in neural network: {e}")

In-place operation error in neural network: a leaf Variable that requires grad is being used in an in-place operation.


Performing the forward pass with `torch.no_grad()` will not track gradients and we will not be able to compute the gradients of the loss with respect to the model parameters. So calling `backward()` will raise an error.

In [9]:
# Using torch.no_grad() to disable gradient tracking when we want to compute gradients
with torch.no_grad():
    output_no_grad = model(data)
try:
    output_no_grad.backward()
except RuntimeError as e:
    print(f"Error when backpropagating after using torch.no_grad(): {e}")

Error when backpropagating after using torch.no_grad(): element 0 of tensors does not require grad and does not have a grad_fn


In [10]:
intermediate = model[0](data).detach()  # Detaching output of the first layer
output_broken = model[2](nn.functional.relu(intermediate))  # Forward through remaining layers

model.zero_grad() # Set gradients to None
# This example does not throw an error because no gradient computation is attempted.
# However, the computational graph is broken, and gradients will not flow back to the detached tensor.
output_broken.backward()  # This will compute gradients only for the second part of the graph.

print("Gradients after detaching intermediate tensor:")
for name, param in model.named_parameters():
    print(f"Gradient for {name}: {param.grad}")


Gradients after detaching intermediate tensor:
Gradient for 0.weight: None
Gradient for 0.bias: None
Gradient for 2.weight: tensor([[0.7499, 0.1019, 0.5888]])
Gradient for 2.bias: tensor([1.])


### Good to Know

- Access **trainable parameter values** using `param.data` only when you are sure you want to bypass autograd. Prefer `with torch.no_grad():` or `.detach()` for safe manipulation.
- Access **gradients** using `param.grad` for individual parameters. Be aware that gradients are `None` until `.backward()` is called.
- `model.layer_name.weight` is a `torch.nn.Parameter` (not a regular `torch.Tensor`) and comes with `requires_grad=True` by default. Parameters are tracked by optimizers for gradient updates.
- Calling `model.zero_grad()` will reset all gradients to `None` in newer versions of PyTorch. To set them to zero, you need to pass `set_to_none=False`.
- PyTorch tracks all operations involving tensors with `requires_grad=True` to build the computational graph dynamically.
- **Contagious gradients**: If `a` requires gradients, any operation involving `a` will produce a tensor that also requires gradients. For example:
  ```python
  a = torch.tensor(5.0, requires_grad=True)
  b = a + 3  # b.requires_grad = True
  ```
- Avoid **in-place operations** (e.g., `a += b` or `tensor.add_(value)`), as they can disrupt autograd's ability to track computations. Instead, use out-of-place equivalents:
  ```python
  tensor = tensor + value
  ```
- Avoid converting tensors to **NumPy arrays** (`.numpy()`) or Python lists (`.tolist()`) if the tensor requires gradients, as it will detach the data from the computational graph. Prefer PyTorch operations to retain gradient tracking.

#### Copying Weights from One Model to Another
If you need to copy the weights of a layer from one model to another, you can use `with torch.no_grad()` to avoid disrupting gradient tracking:
```python
with torch.no_grad():
    model2.layer_name2.weight.copy_(model1.layer_name1.weight)
```
This ensures the weights are copied safely without modifying the computational graph.
```